<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Disease Processor extraction class - Dev notebook

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

In [ ]:
print(manager.sfd_list.columns)

In [ ]:
#print(manager.sfd_list.columns)
unique_values = manager.sfd_list['field.farm.grower.firstname'].unique()
print(unique_values)


### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 -Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from disease_functions.py**

### 🗺️ Configure Disease Extractor

In [ ]:
# Import your class
from earthdaily.agriculture.processors.processor_disease_risk_functions import DiseaseExtractor
extractor = DiseaseExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id", "start_date": "sowingDate"}

extractor.setup_disease_parameters(
    partial_frequency=50,
    exclude_columns=[],
    column_mapping=column_mapping
)

### 🗺️ Test functions

In [ ]:
# Prepare test seasonfield_data
seasonfield_data = {
    "id": "",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "SOYBEANS",
    "sowingDate":"2025-06-01",
    "end_date":"2025-10-01"
}


#### Test get_disease_api

In [ ]:
#============================================================================
# TEST CASE 1: API Call - Raw JSON Response
# ============================================================================
print("\n" + "="*70)
print("TEST CASE 1: Get Disease Data - Raw JSON Response")
print("="*70)

try:
    result = extractor.get_disease_data(entity_data=seasonfield_data)
    
    print("✅ Disease data retrieved successfully!")
    
    # Handle different response formats
    if isinstance(result, list):
        records = result
        print(f"📊 Response type: List with {len(records)} records")
    elif isinstance(result, dict):
        if 'value' in result:
            records = result['value']
            print(f"📊 Response type: Dict with 'value' key - {len(records)} records")
        elif 'data' in result:
            records = result['data']
            print(f"📊 Response type: Dict with 'data' key - {len(records)} records")
        elif 'results' in result:
            records = result['results']
            print(f"📊 Response type: Dict with 'results' key - {len(records)} records")
        else:
            records = [result]
            print(f"📊 Response type: Single dict record")
    else:
        print(f"⚠️ Unexpected response type: {type(result)}")
        print(f"Response: {result}")
        records = []
    
    # Display first few records
    if records and len(records) > 0:
        print(f"\n🔬 First disease record:")
        first_record = records[0]
        for key, value in first_record.items():
            if isinstance(value, dict):
                print(f"  {key}:")
                for sub_key, sub_val in value.items():
                    print(f"    {sub_key}: {sub_val}")
            else:
                print(f"  {key}: {value}")
        
        if len(records) > 1:
            print(f"\n📈 Sample of additional records (showing dates):")
            for i, record in enumerate(records[1:6], start=2):
                print(f"  Record {i}: {record.get('date', 'N/A')}")
            
            if len(records) > 6:
                print(f"  ... and {len(records) - 6} more records")
    else:
        print("⚠️ No records found in response")
    
except Exception as e:
    print(f"❌ API call failed: {e}")
    import traceback
    traceback.print_exc()



In [ ]:
# ============================================================================
# TEST CASE 2: Test with Invalid Data
# ============================================================================
print("\n" + "="*70)
print("TEST CASE 5: Error Handling - Invalid Dates")
print("="*70)

invalid_data = seasonfield_data.copy()
invalid_data['start_date'] = "2025-13-45"  # Invalid date

try:
    result = extractor.get_disease_data_safe(entity_data=invalid_data)
    
    if result['success']:
        print("⚠️ Expected error but got success")
    else:
        print(f"✅ Error handled correctly: {result['error']}")
        
except Exception as e:
    print(f"✅ Exception caught as expected: {e}")





In [ ]:
# ============================================================================
# TEST CASE 3: Test with Missing Geometry
# ============================================================================
print("\n" + "="*70)
print("TEST CASE 6: Error Handling - Missing Geometry")
print("="*70)

invalid_data = seasonfield_data.copy()
invalid_data['geometry'] = None

try:
    result = extractor.get_disease_data_safe(entity_data=invalid_data)
    
    if result['success']:
        print("⚠️ Expected error but got success")
    else:
        print(f"✅ Error handled correctly: {result['error']}")
        
except Exception as e:
    print(f"✅ Exception caught as expected: {e}")

#### Test get_disease_data_safe

In [ ]:
# ============================================================================
# TEST CASE 4: Safe Wrapper - Structured Response
# ============================================================================
print("\n" + "="*70)
print("TEST CASE 2: Get Disease Data (Safe) - Error Handling")
print("="*70)

try:
    safe_result = extractor.get_disease_data_safe(entity_data=seasonfield_data)
    
    print(f"Success: {safe_result['success']}")
    print(f"Entity ID: {safe_result['entity_id']}")
    
    if safe_result['success']:
        print("✅ Disease data retrieved successfully via safe wrapper!")
        data = safe_result['data']
        
        # Count records
        if isinstance(data, list):
            print(f"📊 Retrieved {len(data)} disease records")
        elif isinstance(data, dict):
            if 'value' in data:
                print(f"📊 Retrieved {len(data['value'])} disease records")
            elif 'data' in data:
                print(f"📊 Retrieved {len(data['data'])} disease records")
    else:
        print(f"❌ Error: {safe_result['error']}")
        
except Exception as e:
    print(f"❌ Safe wrapper failed: {e}")
    import traceback
    traceback.print_exc()

#### Test format_disease_json

In [ ]:
# ============================================================================
# TEST CASE 5: Format Disease JSON to DataFrame
# ============================================================================
print("\n" + "="*70)
print("TEST CASE 3: Format Disease JSON to DataFrame")
print("="*70)

try:
    # Get raw JSON
    raw_json = extractor.get_disease_data(entity_data=seasonfield_data)
    
    # Format to DataFrame
    disease_df = extractor.format_disease_json(raw_json, entity_data=seasonfield_data)
    
    if disease_df is not None and not disease_df.empty:
        print(f"✅ Formatted disease data into DataFrame!")
        print(f"\n📊 DataFrame Info:")
        print(f"   Shape: {disease_df.shape[0]} rows × {disease_df.shape[1]} columns")
        print(f"   Date range: {disease_df['date'].min()} to {disease_df['date'].max()}")
        
        print(f"\n📋 Columns ({len(disease_df.columns)}):")
        for col in disease_df.columns:
            dtype = disease_df[col].dtype
            non_null = disease_df[col].notna().sum()
            print(f"   {col}: {dtype} ({non_null} non-null)")
        
        print(f"\n🔬 First 3 rows:")
        print(disease_df.head(3).to_string())
        
        print(f"\n📈 Summary statistics (numeric columns only):")
        numeric_cols = disease_df.select_dtypes(include=['number']).columns
        if len(numeric_cols) > 0:
            print(disease_df[numeric_cols].describe().to_string())
        else:
            print("   No numeric columns found")
            
    else:
        print("⚠️ No data returned from formatting")
        
except Exception as e:
    print(f"❌ Formatting failed: {e}")
    import traceback
    traceback.print_exc()



### 🗺️ process_single_entity

In [ ]:
# ============================================================================
# TEST CASE 6: Process Single Entity
# ============================================================================
print("\n" + "="*70)
print("TEST CASE 4: Process Single Entity - Complete Pipeline")
print("="*70)

try:
    # Process single entity
    result = extractor.process_single_entity_disease(seasonfield_data)
    
    df = result.get("data")
    error = result.get("error")
    
    if error:
        print(f"❌ Processing error: {error}")
    
    if df is not None and not df.empty:
        print(f"✅ Single entity processed successfully!")
        print(f"\n📊 Result DataFrame:")
        print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        
        # Show entity metadata columns
        metadata_cols = [col for col in df.columns if col not in ['date'] and not col.startswith(('temperature', 'precipitation', 'wind', 'humidity'))]
        if metadata_cols:
            print(f"\n🏷️ Entity Metadata Columns:")
            for col in metadata_cols[:10]:  # Show first 10
                print(f"   {col}: {df[col].iloc[0]}")
            if len(metadata_cols) > 10:
                print(f"   ... and {len(metadata_cols) - 10} more metadata columns")
        
        # Show disease data columns
        disease_cols = [col for col in df.columns if col not in metadata_cols and col != 'date']
        if disease_cols:
            print(f"\n🦠 Disease Data Columns ({len(disease_cols)}):")
            for col in disease_cols:
                print(f"   {col}")
        
        print(f"\n🔬 Sample rows:")
        print(df.head(3).to_string())
        
    else:
        print("⚠️ No data returned from processing")
        
except Exception as e:
    print(f"❌ Processing failed: {e}")
    import traceback
    traceback.print_exc()



# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*70)
print("TEST SUITE COMPLETE")
print("="*70)
print("\n✅ All test cases executed. Review results above for any failures.")

### Test historical years validation (list, string, column_mapping)

Validates that `years` works as a native list, comma-separated string (pipeline flattening),
and via `column_mapping` remapping from `historical_seasons`.

In [ ]:
from earthdaily.agriculture.core.api_utils import validate_historical_years

# Test validate_historical_years directly
test_cases = [
    None,
    'ALL',
    [2024, 2023, 2022],
    '2024,2023,2022',
    5,
]

for tc in test_cases:
    result = validate_historical_years(tc)
    print(f'  {str(tc):30s} -> {result} ({type(result).__name__})')


### 🗺️ process_disease_bulk_extraction_parallel

In [ ]:
import pandas as pd
top25 = manager.sfd_list.head(50)

# Convert sowingDate to datetime and compute end_date
top25['sowingDate'] = pd.to_datetime(top25['sowingDate'])
top25['end_date'] = top25['sowingDate'] + pd.Timedelta(days=50)

# Launch extraction with 20 threads 

result = extractor.process_entity_disease_bulk_parallel(
    entity_list=top25,
    params=None,
    max_workers=10,
    output_path=manager.output_result_dir,
    fail_safe=False,
    filter_column="crop.id",
    filter_value="CORN",
    filter_type="exclude", # filter type used to 'include' or 'exclude' row matching column and value filter
    prefix= "disease"
)

print(f"\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")

print("\n🔍 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")

In [ ]:
# Get the clean DataFrame
results=result["results_df"]
print(results.columns)

In [ ]:
print(results.head)
